# Why Rerank? [Step 1 - Recall First, Precision Second]

> **MLCourse - Agentic AI - Advanced RAG - Reranking**

Every retrieval system you have built so far in this track ends the same way:
you embed the query, you pull the top *k* nearest chunks, and you hand them to
the LLM. That last step is where most RAG systems quietly lose quality.

The reason is that the retriever you used was optimised for **speed at scale**,
not for **judging relevance**. This notebook explains the difference between
the two model families involved - the **bi-encoder** and the **cross-encoder** -
and introduces the two-stage *retrieve-then-rerank* pattern that is the single
most standard quality upgrade in production RAG.

By the end of this notebook you will be able to explain, with numbers from your
own corpus, why a vector search that "looks fine" still puts mediocre chunks in
position 1.

### 1. Setup

Every notebook in this module is self-contained. We load `GROQ_API_KEY` from
`03_agentic_ai/.env` by walking up the directory tree from wherever the notebook
happens to be opened, so the same code works no matter your working directory.

In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


### The model, and the documented fallback

We use **Groq** with `qwen/qwen3.8-27b` throughout this module. Groq's free tier
allows roughly **8000 tokens per minute**, so every loop below paces itself and
retries with exponential backoff instead of hammering the endpoint.

If you are offline or rate-limited, a local Ollama server at
`http://localhost:11434` is the drop-in replacement - swap the two lines below
and nothing else in the notebook changes:

```python
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.1:8b", temperature=0)
```

In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


### 2. The corpus

We reuse *Alice in Wonderland* from `03_agentic_ai/data/`, chunked into
paragraphs exactly as in [`../01_hybrid_search`](../01_hybrid_search/README.md).
Using the same corpus and the same chunking is deliberate: it means any quality
difference you measure later comes from the reranker, not from a data change.

In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

# Paragraph-sized chunks: human-readable units, good enough for retrieval demos
# and identical to the chunking used in ../01_hybrid_search.
paragraphs = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]

print("characters :", len(raw_text))
print("paragraphs :", len(paragraphs))
print("example    :", paragraphs[10][:150], "...")

characters : 144696
paragraphs : 237
example    : Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passa ...


### 3. Bi-encoders: fast, but the query and document never meet

A **bi-encoder** is what powers ordinary vector search. It encodes the query and
each document **separately** into fixed vectors, then compares them with a
cosine similarity:

```
        query  ->  [encoder]  ->  q  (384 numbers)
     document  ->  [encoder]  ->  d  (384 numbers)
                                score = q . d
```

The document vectors can be computed **once, offline**, and stored in a vector
index. At query time you encode one short string and do a nearest-neighbour
lookup. That is why vector search over a million documents takes milliseconds.

The price you pay: the document was compressed into 384 numbers *without ever
knowing what would be asked of it*. All the query can do is find a vector that
happens to point in a similar direction. Nuance - negation, which entity does
what to whom, whether the passage actually *answers* the question - is mostly
lost in that compression.

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# The whole corpus is embedded ONCE. This is the offline, amortised cost.
t0 = time.time()
doc_vectors = bi_encoder.encode(paragraphs, normalize_embeddings=True,
                                batch_size=64, show_progress_bar=False)
encode_time = time.time() - t0

print(f"embedded {len(paragraphs)} paragraphs in {encode_time:.2f}s")
print("vector shape:", doc_vectors.shape)
print(f"per-document cost: {1000 * encode_time / len(paragraphs):.2f} ms (paid once, offline)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedded 237 paragraphs in 2.83s
vector shape: (237, 384)
per-document cost: 11.93 ms (paid once, offline)


In [5]:
QUERY = "What advice does the Caterpillar give Alice?"

t0 = time.time()
qv = bi_encoder.encode([QUERY], normalize_embeddings=True)[0]
sims = doc_vectors @ qv                      # cosine, since everything is normalised
dense_order = list(np.argsort(sims)[::-1])
search_ms = 1000 * (time.time() - t0)

print(f"query: {QUERY}")
print(f"search over {len(paragraphs)} documents took {search_ms:.1f} ms\n")
for rank, doc_id in enumerate(dense_order[:5], 1):
    print(f"#{rank}  cos={sims[doc_id]:.3f}  doc_{doc_id}")
    print("     ", paragraphs[doc_id][:160], "...")

query: What advice does the Caterpillar give Alice?
search over 237 documents took 22.2 ms

#1  cos=0.674  doc_91
      Which brought them back again to the beginning of the conversation. Alice felt a little irritated at the Caterpillar’s making such _very_ short remarks, and she ...
#2  cos=0.642  doc_93
      This time Alice waited patiently until it chose to speak again. In a minute or two the Caterpillar took the hookah out of its mouth and yawned once or twice, an ...
#3  cos=0.588  doc_90
      “Well, perhaps you haven’t found it so yet,” said Alice; “but when you have to turn into a chrysalis—you will some day, you know—and then after that into a butt ...
#4  cos=0.576  doc_86
      “And yet what a dear little puppy it was!” said Alice, as she leant against a buttercup to rest herself, and fanned herself with one of the leaves: “I should ha ...
#5  cos=0.560  doc_190
      So Alice began telling them her adventures from the time when she first saw the White Rabbit. She was a lit

Look carefully at those five results before moving on.

They are all *about* Alice, and several mention the Caterpillar, which is why
they scored well. But ask yourself the question a user would ask: **does the
paragraph in position 1 actually answer "what advice did the Caterpillar
give"?** In most runs it does not - the top hit is topically adjacent rather
than answer-bearing.

Notice also how **flat** the cosine scores are. The gap between #1 and #5 is
usually a few hundredths. The bi-encoder is telling you "these five are all
roughly equally on-topic" - which is exactly the information it has, and
exactly not the information you need.

### 4. Cross-encoders: slow, but the query and document are read together

A **cross-encoder** takes a different shape entirely. Instead of two separate
vectors, it concatenates the pair and pushes them through a transformer as a
single input:

```
   [CLS] query [SEP] document [SEP]  ->  [transformer]  ->  one relevance score
```

Because every query token can attend to every document token, the model can
represent things a dot product simply cannot: that the document *denies* the
premise, that the entity mentioned is a different one, that the passage
mentions the topic but never states the fact being asked for.

The price: there is **no precomputation**. The score depends on the pair, so you
must run the model once per (query, document) pair at query time. Scoring a
million documents this way is impossible. Scoring the **top 50 from a fast
retriever** is easy - and that is the whole idea.

In [6]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("cross-encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2")

# Score the SAME five paragraphs the bi-encoder ranked, as (query, doc) pairs.
candidates = dense_order[:5]
pairs = [(QUERY, paragraphs[i]) for i in candidates]

t0 = time.time()
ce_scores = cross_encoder.predict(pairs)
ce_ms = 1000 * (time.time() - t0)

print(f"scored {len(pairs)} pairs in {ce_ms:.0f} ms "
      f"({ce_ms / len(pairs):.0f} ms per pair)\n")

for doc_id, score in sorted(zip(candidates, ce_scores), key=lambda x: -x[1]):
    print(f"cross-encoder={score:+7.3f}  bi-encoder cos={sims[doc_id]:.3f}  doc_{doc_id}")
    print("     ", paragraphs[doc_id][:150], "...")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

cross-encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2
scored 5 pairs in 75 ms (15 ms per pair)

cross-encoder= +2.152  bi-encoder cos=0.642  doc_93
      This time Alice waited patiently until it chose to speak again. In a minute or two the Caterpillar took the hookah out of its mouth and yawned once or ...
cross-encoder= +2.143  bi-encoder cos=0.674  doc_91
      Which brought them back again to the beginning of the conversation. Alice felt a little irritated at the Caterpillar’s making such _very_ short remark ...
cross-encoder= -0.522  bi-encoder cos=0.560  doc_190
      So Alice began telling them her adventures from the time when she first saw the White Rabbit. She was a little nervous about it just at first, the two ...
cross-encoder= -5.892  bi-encoder cos=0.588  doc_90
      “Well, perhaps you haven’t found it so yet,” said Alice; “but when you have to turn into a chrysalis—you will some day, you know—and then after that i ...
cross-encoder= -7.942  bi-encoder cos=0.576  d

Two things to take away from that output.

**The scores are no longer flat.** Cross-encoder scores here are raw logits, so
they are unbounded and often span from about `-11` (clearly irrelevant) to about
`+10` (a direct answer). A spread of twenty points is a *decision*, not a
tie-break. The bi-encoder's few-hundredths spread was not.

**The order changed.** The paragraph the bi-encoder ranked first is usually not
the one the cross-encoder ranks first. Both models are "right" about what they
measure - the bi-encoder measures topical proximity, the cross-encoder measures
whether this document answers this query.

### 5. Why not just use the cross-encoder for everything?

Because of the number below. Let us measure what it would cost to cross-encode
the entire corpus - which is what "no first stage" would mean.

In [7]:
t0 = time.time()
_ = cross_encoder.predict([(QUERY, paragraphs[i]) for i in range(60)])
per_pair_ms = 1000 * (time.time() - t0) / 60

print(f"measured cross-encoder cost : {per_pair_ms:.1f} ms per (query, document) pair\n")
for corpus_size in [237, 10_000, 1_000_000]:
    full_s = per_pair_ms * corpus_size / 1000
    print(f"  corpus of {corpus_size:>9,} docs -> {full_s:>10,.1f} s per query if we score everything")

print(f"\n  rerank only the top 50 from a fast retriever -> "
      f"{per_pair_ms * 50 / 1000:.2f} s per query, at ANY corpus size")

measured cross-encoder cost : 15.7 ms per (query, document) pair

  corpus of       237 docs ->        3.7 s per query if we score everything
  corpus of    10,000 docs ->      156.7 s per query if we score everything
  corpus of 1,000,000 docs ->   15,665.2 s per query if we score everything

  rerank only the top 50 from a fast retriever -> 0.78 s per query, at ANY corpus size


That last line is the entire argument for the two-stage pattern. The cost of
reranking does not grow with your corpus - it grows with your **candidate
count**, which you choose.

### 6. The retrieve-then-rerank pattern

```
   query
     |
     v
  +--------------------------+   cheap, O(log N) with an index
  |  STAGE 1: RETRIEVE       |   bi-encoder / BM25 / hybrid RRF
  |  goal: RECALL            |   return top 50 candidates
  +--------------------------+
     |
     v  50 candidates
  +--------------------------+   expensive, O(candidates)
  |  STAGE 2: RERANK         |   cross-encoder scores each pair
  |  goal: PRECISION         |   return the best 5
  +--------------------------+
     |
     v  5 chunks
   LLM generation
```

Each stage has one job, and the jobs are different:

- **Stage 1 optimises recall.** Its only failure mode that matters is *leaving
  the right document out of the candidate set*. A document that is not in the
  50 can never be recovered - the reranker never sees it. So retrieve
  generously: 30-100 candidates is typical, far more than you would ever put in
  a prompt.
- **Stage 2 optimises precision.** It cannot add documents, only reorder them.
  Its job is to make sure that the 3-5 chunks that actually reach the LLM are
  the best 3-5 in the candidate set.

Let us run both stages end-to-end.

In [8]:
def retrieve(query, n_candidates=50):
    """STAGE 1 - cheap bi-encoder recall over the whole corpus."""
    q = bi_encoder.encode([query], normalize_embeddings=True)[0]
    return list(np.argsort(doc_vectors @ q)[::-1][:n_candidates])


def rerank(query, candidate_ids, top_k=5):
    """STAGE 2 - expensive cross-encoder precision over the candidates only."""
    scores = cross_encoder.predict([(query, paragraphs[i]) for i in candidate_ids])
    ordered = sorted(zip(candidate_ids, scores), key=lambda x: -x[1])
    return [(int(i), float(s)) for i, s in ordered[:top_k]]


candidates = retrieve(QUERY, n_candidates=50)
reranked = rerank(QUERY, candidates, top_k=5)

print("STAGE 1 (bi-encoder) top 5:")
for rank, doc_id in enumerate(candidates[:5], 1):
    print(f"  #{rank} doc_{doc_id}: {paragraphs[doc_id][:100]}...")

print("\nSTAGE 2 (after cross-encoder rerank of 50 candidates) top 5:")
for rank, (doc_id, score) in enumerate(reranked, 1):
    moved = candidates.index(doc_id) + 1
    print(f"  #{rank} doc_{doc_id} (was #{moved} in stage 1, score={score:+.2f}): "
          f"{paragraphs[doc_id][:100]}...")

STAGE 1 (bi-encoder) top 5:
  #1 doc_91: Which brought them back again to the beginning of the conversation. Alice felt a little irritated at...
  #2 doc_93: This time Alice waited patiently until it chose to speak again. In a minute or two the Caterpillar t...
  #3 doc_90: “Well, perhaps you haven’t found it so yet,” said Alice; “but when you have to turn into a chrysalis...
  #4 doc_86: “And yet what a dear little puppy it was!” said Alice, as she leant against a buttercup to rest hers...
  #5 doc_190: So Alice began telling them her adventures from the time when she first saw the White Rabbit. She wa...

STAGE 2 (after cross-encoder rerank of 50 candidates) top 5:
  #1 doc_93 (was #2 in stage 1, score=+2.15): This time Alice waited patiently until it chose to speak again. In a minute or two the Caterpillar t...
  #2 doc_91 (was #1 in stage 1, score=+2.14): Which brought them back again to the beginning of the conversation. Alice felt a little irritated at...
  #3 doc_19 (was #39 in 

The `was #N in stage 1` annotation is the interesting column. Documents that
were sitting at rank 20 or 30 - well outside anything you would have put in a
prompt - routinely climb into the top 5 once a model actually reads them
alongside the question.

### 7. Does it change the answer? Ask the LLM both ways.

Reranking is only worth its latency if the *generated answer* improves. Let us
give Groq the same question twice: once with the bi-encoder's top 3, once with
the reranked top 3.

In [9]:
def answer_from(doc_ids, query):
    context = "\n\n".join(f"[doc_{i}] {paragraphs[i]}" for i in doc_ids)
    prompt = (
        "Answer the question using ONLY the context below. "
        "Be specific and quote the relevant detail. "
        "If the context does not contain the answer, say so plainly.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    )
    return ask(prompt)


baseline_ids = candidates[:3]
reranked_ids = [i for i, _ in reranked[:3]]

print("=" * 70)
print("ANSWER FROM BI-ENCODER TOP 3  (docs:", baseline_ids, ")")
print("=" * 70)
print(answer_from(baseline_ids, QUERY))

ANSWER FROM BI-ENCODER TOP 3  (docs: [np.int64(91), np.int64(93), np.int64(90)] )


Based on the context provided, the Caterpillar gives Alice the following advice as it crawls away: “One side will make you grow taller, and the other side will make you grow shorter.”


In [10]:
print("=" * 70)
print("ANSWER FROM RERANKED TOP 3  (docs:", reranked_ids, ")")
print("=" * 70)
print(answer_from(reranked_ids, QUERY))

ANSWER FROM RERANKED TOP 3  (docs: [93, 91, 19] )


Based on the context provided, the Caterpillar gives Alice the following advice: “One side will make you grow taller, and the other side will make you grow shorter.”


### 8. Pitfalls

- **Reranking cannot fix bad recall.** If stage 1 missed the answer, no
  reranker can find it. When quality is poor, always check recall of the
  candidate set *first* - it is the more common failure.
- **Candidate count is the real knob.** Going from 10 to 50 candidates usually
  helps a lot; 50 to 200 usually helps a little and costs four times as much.
  Notebook 05 measures this curve.
- **Cross-encoder scores are not probabilities.** `ms-marco` models emit raw
  logits. Do not put a threshold like `> 0.5` on them - calibrate against your
  own data, or just take the top-k.
- **The reranker has a token limit too.** Long chunks get truncated inside the
  cross-encoder, so a 4000-character chunk may be judged on its first part only.
  This interacts with chunking - see [`../13_contextual_retrieval`](../13_contextual_retrieval/README.md).
- **Do not confuse reranking with fusion.** RRF (from
  [`../01_hybrid_search`](../01_hybrid_search/README.md)) merges *ranked lists* by
  position and never reads the text. A reranker reads the text. They compose
  beautifully - notebook 03 does exactly that.

### 9. Key takeaways

- Bi-encoders encode query and document **separately**: fast, precomputable,
  scales to millions - but relevance judgement is weak and scores are flat.
- Cross-encoders encode the pair **jointly**: much better judgement, no
  precomputation possible, cost is linear in the number of candidates.
- The two-stage **retrieve-then-rerank** pattern uses each where it is strong:
  recall from stage 1, precision from stage 2.
- Reranking cost is independent of corpus size. It depends only on how many
  candidates you choose to score.

Next: [`02_cross_encoder_reranking.ipynb`](02_cross_encoder_reranking.ipynb) goes
deeper on the `CrossEncoder` API - batching, score interpretation, model choice,
and building a reusable reranker class.